# Copy machine-readable results from Drive into the repo and push
Checkpoints/adapters stay on Drive (not committed).

### Setup (run every session)
Results and large artefacts go to Google Drive so a disconnected runtime loses nothing; every step is resumable.
Use an **A100** (or L4) runtime with High-RAM: *Runtime → Change runtime type*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO_URL = 'https://github.com/adilawan1/ATML-PA4.git'   # <- edit once
DRIVE = '/content/drive/MyDrive/atml-pa4'
os.makedirs(DRIVE, exist_ok=True)
os.environ['PA4_RESULTS'] = f'{DRIVE}/results'
os.environ['PA4_CKPT'] = f'{DRIVE}/checkpoints'
os.environ['PA4_DATA'] = f'{DRIVE}/atml-pa4-datasets'   # on Drive so data/local_source_revisions.json persists
if not os.path.exists('/content/atml-pa4'):
    !git clone -q $REPO_URL /content/atml-pa4
%cd /content/atml-pa4
!git pull -q
!pip install -q -r requirements.txt
!bash scripts/get_data.sh
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python scripts/record_env.py | tail -1

In [ ]:
!rsync -a --exclude '*.pt' --exclude '*.safetensors' $PA4_RESULTS/ results/
!du -sh results; git status --short | head -30

In [ ]:
# Push (needs a GitHub token with repo scope, e.g. stored as a Colab secret named GH_TOKEN)
from google.colab import userdata
import subprocess
tok = userdata.get('GH_TOKEN')
!git config user.email "you@example.com"; git config user.name "Your Name"
!git add results && git commit -qm "Add experiment results" || echo nothing to commit
remote = REPO_URL.replace('https://', f'https://{tok}@')
subprocess.run(['git', 'push', remote, 'HEAD:main'], check=True)